# Entrenar modelo de clasificación de imágenes

Notebook interactivo para entrenar un clasificador de imágenes.
Todo el código es visible y modificable.

| Paso | Descripción |
|------|-------------|
| 1 | Setup (imports + seeds) |
| 2 | Configurar y cargar dataset |
| 3 | Data Augmentation |
| 4 | Construir modelo (CNN custom o MobileNetV2) |
| 5 | Entrenar |
| 6 | Evaluar |
| 7 | Exportar modelo .h5 |

> Ejecute las celdas en orden. Modifique los valores para experimentar.

---
## 1. Setup

In [ ]:
import os, random, shutil, glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output

# Semillas para reproducibilidad
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow {tf.__version__} — Listo")

---
## 2. Configurar y cargar dataset

Use los widgets para configurar la ruta, resolución y split train/val.
Luego haga clic en **Cargar dataset**.

In [ ]:
# --- Widgets de configuración ---
w_path = widgets.Text(
    value='../data/mi_dataset', placeholder='Ruta a carpeta con subcarpetas por clase',
    description='Dataset:', layout=widgets.Layout(width='500px')
)
w_size = widgets.Dropdown(
    options=[64, 128, 224], value=128, description='Resolución:'
)
w_split = widgets.FloatSlider(
    value=0.8, min=0.5, max=0.95, step=0.05,
    description='Train/Val:', readout_format='.0%'
)
w_btn = widgets.Button(
    description=' Cargar dataset', button_style='primary', icon='folder-open'
)
w_out = widgets.Output()

# --- Estado global del notebook ---
cfg = {
    'dataset_dir': '', 'split_dir': '', 'class_names': [],
    'img_size': 128, 'num_classes': 0,
    'train_gen': None, 'val_gen': None, 'model': None, 'history': None,
}

def _show_dataset_info(ddir, img_size):
    """Muestra resumen del dataset."""
    clases = sorted([d for d in os.listdir(ddir)
                     if os.path.isdir(os.path.join(ddir, d))])
    if len(clases) < 2:
        print(f"\u2718 Se necesitan al menos 2 subcarpetas. Encontradas: {clases}")
        return False

    counts = {}
    for c in clases:
        imgs = [f for f in os.listdir(os.path.join(ddir, c))
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        counts[c] = len(imgs)

    total = sum(counts.values())
    cfg['dataset_dir'] = ddir
    cfg['class_names'] = clases
    cfg['img_size'] = img_size
    cfg['num_classes'] = len(clases)

    print(f"\u2714 Dataset: {ddir}")
    print(f"  Clases: {clases}")
    print(f"  Resolución: {img_size}\u00d7{img_size}")
    print(f"  Total: {total} imágenes\n")

    max_count = max(counts.values()) if counts else 1
    for c, n in counts.items():
        bar = '\u2588' * max(1, n * 30 // max_count)
        print(f"  {c:>15s} {bar} {n}")

    if counts:
        vals = list(counts.values())
        if max(vals) > min(vals) * 2:
            print(f"\n  \u26a0 Dataset desbalanceado. Considere capturar más imágenes de las clases pequeñas.")

    # Preview
    n_cols = min(4, len(clases))
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
    if n_cols == 1: axes = [axes]
    for ax, c in zip(axes, clases[:n_cols]):
        cdir = os.path.join(ddir, c)
        imgs = [f for f in os.listdir(cdir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if imgs:
            ax.imshow(load_img(os.path.join(cdir, random.choice(imgs)),
                               target_size=(img_size, img_size)))
        ax.set_title(f'{c} ({counts[c]})', fontsize=12, fontweight='bold')
        ax.axis('off')
    plt.suptitle('Vista previa', fontsize=14)
    plt.tight_layout(); plt.show()
    return True

def _split_dataset(ddir, split_dir, names, ratio):
    """Divide dataset en train/validation."""
    if os.path.exists(os.path.join(split_dir, 'train')):
        shutil.rmtree(split_dir)
    random.seed(42)
    for c in names:
        imgs = sorted([f for f in os.listdir(os.path.join(ddir, c))
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        random.shuffle(imgs)
        n = int(len(imgs) * ratio)
        for sub, lst in [('train', imgs[:n]), ('validation', imgs[n:])]:
            d = os.path.join(split_dir, sub, c)
            os.makedirs(d, exist_ok=True)
            for f in lst:
                shutil.copy(os.path.join(ddir, c, f), os.path.join(d, f))

def _cargar(_):
    with w_out:
        clear_output(wait=True)
        ddir = w_path.value.strip()
        if not os.path.isdir(ddir):
            print(f"\u2718 No se encontró: {ddir}")
            return
        if not _show_dataset_info(ddir, w_size.value):
            return

        # Split train/validation
        split_dir = ddir.rstrip('/') + '_split'
        cfg['split_dir'] = split_dir
        ratio = w_split.value
        print(f"\nDividiendo: {ratio:.0%} train / {1-ratio:.0%} validation...")
        _split_dataset(ddir, split_dir, cfg['class_names'], ratio)

        # Contar split
        for sub in ['train', 'validation']:
            n = sum(len(os.listdir(os.path.join(split_dir, sub, c)))
                    for c in cfg['class_names'])
            print(f"  {sub}: {n} imágenes")
        print("\n\u2714 Dataset listo. Continue con la celda 3.")

w_btn.on_click(_cargar)
display(widgets.VBox([w_path, w_size, w_split, w_btn, w_out]))

---
## 3. Data Augmentation

El Data Augmentation genera variaciones de las imágenes de entrenamiento
(rotaciones, desplazamientos, cambios de brillo, zoom) para que el modelo
generalice mejor.

**Modifique los valores** y observe cómo cambian las imágenes aumentadas.

> **Nota**: El preprocesamiento depende del modelo que va a usar:
> - **CNN custom**: `rescale=1./255` (normaliza a [0, 1])
> - **MobileNetV2**: `preprocessing_function` de Keras (normaliza a [-1, 1])

In [ ]:
# ============================================================
# ELEGIR MODELO — cambie esta variable
# ============================================================
# Opciones: 'cnn' o 'mobilenet'
#   - cnn:       Entrena desde cero. Bueno para aprender, puede quedar corto.
#   - mobilenet: Transfer learning (MobileNetV2 pre-entrenado en ImageNet).
#                Mejor accuracy con pocos datos. Recomendado para producción.

MODELO = 'mobilenet'  # <-- cambiar a 'cnn' para entrenar desde cero

# ============================================================
# DATA AUGMENTATION — modifique estos valores para experimentar
# ============================================================

sz = cfg['img_size']

if MODELO == 'mobilenet':
    # MobileNetV2 necesita su propio preprocesamiento ([-1, 1])
    train_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_preprocess,
        rotation_range=20,
        width_shift_range=0.1,
        height_shift_range=0.1,
        brightness_range=[0.8, 1.2],
        zoom_range=0.1,
        fill_mode='nearest'
    )
    val_datagen = ImageDataGenerator(
        preprocessing_function=mobilenet_preprocess
    )
    print("Preprocesamiento: MobileNetV2 ([-1, 1])")
else:
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=20,          # grados de rotación aleatoria
        width_shift_range=0.1,      # desplazamiento horizontal (fracción)
        height_shift_range=0.1,     # desplazamiento vertical (fracción)
        brightness_range=[0.8, 1.2],# rango de brillo
        zoom_range=0.1,             # zoom aleatorio
        fill_mode='nearest'         # cómo rellenar píxeles nuevos
    )
    val_datagen = ImageDataGenerator(
        rescale=1./255  # validación: solo normalizar, sin augmentation
    )
    print("Preprocesamiento: rescale 1/255 ([0, 1])")

# --- Crear generadores ---
train_gen = train_datagen.flow_from_directory(
    os.path.join(cfg['split_dir'], 'train'),
    target_size=(sz, sz), batch_size=32,
    class_mode='categorical', shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    os.path.join(cfg['split_dir'], 'validation'),
    target_size=(sz, sz), batch_size=32,
    class_mode='categorical', shuffle=False
)

cfg['train_gen'] = train_gen
cfg['val_gen'] = val_gen
cfg['modelo_tipo'] = MODELO

num_classes = len(train_gen.class_indices)
cfg['num_classes'] = num_classes
cfg['class_names'] = list(train_gen.class_indices.keys())

print(f"Clases: {cfg['class_names']}")
print(f"Train: {train_gen.samples} | Validation: {val_gen.samples}")

# --- Visualizar ejemplos augmentados ---
print("\nEjemplos de Data Augmentation:")

# Para visualización, necesitamos desnormalizar
sample_batch, _ = next(train_gen)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, ax in enumerate(axes.ravel()):
    if i < len(sample_batch):
        img = sample_batch[i]
        # Desnormalizar para mostrar
        if MODELO == 'mobilenet':
            img = (img + 1) / 2  # [-1,1] → [0,1]
        ax.imshow(np.clip(img, 0, 1))
    ax.axis('off')
plt.suptitle('Imágenes del batch (con augmentation aplicado)', fontsize=14)
plt.tight_layout(); plt.show()

train_gen.reset()

---
## 4. Construir modelo

Elija entre dos arquitecturas:

- **CNN custom**: 3 bloques Conv2D + BatchNorm + MaxPool. Entrena desde cero.
  Bueno para aprender cómo funciona una CNN internamente.
- **MobileNetV2**: Red pre-entrenada en ImageNet (1.4M imágenes, 1000 clases).
  Se congela la base y solo se entrena el clasificador final (transfer learning).
  Recomendado cuando el dataset es pequeño o se necesita mejor accuracy.

In [ ]:
# ============================================================
# CONSTRUIR MODELO — depende de la variable MODELO (celda 3)
# ============================================================

if MODELO == 'mobilenet':
    # ── MobileNetV2 (transfer learning) ──
    # Base pre-entrenada: se congela para no destruir lo que ya aprendió
    base = MobileNetV2(
        weights='imagenet',
        include_top=False,  # sin la capa final de ImageNet
        input_shape=(sz, sz, 3)
    )
    base.trainable = False  # congelar pesos del base

    # Clasificador nuevo encima del base
    inp = layers.Input(shape=(sz, sz, 3))
    x = base(inp, training=False)  # training=False: BatchNorm en modo inferencia
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=inp, outputs=out)

    print(f"MobileNetV2: base congelada ({base.count_params():,} params)")
    print(f"Clasificador: {model.count_params() - base.count_params():,} params entrenables")

else:
    # ── CNN custom (desde cero) ──
    model = models.Sequential()
    model.add(layers.Input(shape=(sz, sz, 3)))

    # Bloque 1: 32 filtros
    model.add(layers.Conv2D(32, 3, padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2))

    # Bloque 2: 64 filtros
    model.add(layers.Conv2D(64, 3, padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2))

    # Bloque 3: 128 filtros
    model.add(layers.Conv2D(128, 3, padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2))

    # Clasificador
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    print("CNN custom: 3 bloques Conv2D")

model.summary()

---
## 5. Entrenar

- **CNN custom**: Entrenamiento directo con EarlyStopping y ReduceLROnPlateau.
- **MobileNetV2**: Dos fases:
  1. **Base congelada** — solo entrena el clasificador (pocas épocas, LR normal)
  2. **Fine-tuning** — descongela las últimas capas del base (más épocas, LR bajo)

In [ ]:
# ============================================================
# COMPILAR Y ENTRENAR
# ============================================================

# Callback para mostrar progreso
class PrintProgress(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        ta = logs.get('accuracy', 0)
        va = logs.get('val_accuracy', 0)
        print(f"  Época {epoch+1:2d}: train={ta:.1%}  val={va:.1%}", flush=True)

if MODELO == 'mobilenet':
    # ── Fase 1: base congelada, entrenar solo el clasificador ──
    EPOCHS_FASE1 = 5
    LR_FASE1 = 0.001

    model.compile(
        optimizer=Adam(learning_rate=LR_FASE1),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    print(f"Fase 1: clasificador ({EPOCHS_FASE1} épocas, LR={LR_FASE1})\n")

    h1 = model.fit(
        train_gen, epochs=EPOCHS_FASE1, validation_data=val_gen, verbose=0,
        callbacks=[
            EarlyStopping(patience=5, restore_best_weights=True),
            PrintProgress()
        ]
    )

    # ── Fase 2: fine-tuning (descongelar últimas 30 capas del base) ──
    EPOCHS_FASE2 = 25
    LR_FASE2 = 0.00001  # LR mucho menor para no destruir pesos pre-entrenados

    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=Adam(learning_rate=LR_FASE2),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    capas_entrenables = sum(1 for l in base.layers if l.trainable)
    print(f"\nFase 2: fine-tuning ({capas_entrenables} capas descongeladas, "
          f"{EPOCHS_FASE2} épocas, LR={LR_FASE2})\n")

    h2 = model.fit(
        train_gen, epochs=EPOCHS_FASE2, validation_data=val_gen, verbose=0,
        callbacks=[
            EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(factor=0.5, patience=5, verbose=1),
            PrintProgress()
        ]
    )

    # Combinar historias de ambas fases
    hist = {}
    for key in h1.history:
        hist[key] = h1.history[key] + h2.history[key]

else:
    # ── CNN custom: entrenamiento directo ──
    LR = 0.001
    EPOCHS = 30

    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    print(f"Entrenando CNN ({num_classes} clases, {sz}px, LR={LR})...\n")

    h = model.fit(
        train_gen, epochs=EPOCHS, validation_data=val_gen, verbose=0,
        callbacks=[
            EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(factor=0.5, patience=5, verbose=1),
            PrintProgress()
        ]
    )
    hist = h.history

cfg['model'] = model
cfg['history'] = hist

# --- Resultados ---
ne = len(hist['accuracy'])
ta = hist['accuracy'][-1]
va = hist['val_accuracy'][-1]
print(f"\n\u2714 Entrenamiento completo ({ne} épocas)")
print(f"  Train accuracy: {ta:.1%}")
print(f"  Val accuracy:   {va:.1%}")
if ta - va > 0.15:
    print(f"  \u26a0 Posible overfitting (diferencia {ta-va:.1%})")

# --- Gráficas ---
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
r = range(1, ne + 1)
a1.plot(r, hist['accuracy'], 'b-', label='Train')
a1.plot(r, hist['val_accuracy'], 'r-', label='Val')
a1.set_title('Accuracy'); a1.set_xlabel('Época')
a1.legend(); a1.grid(alpha=0.3)
a2.plot(r, hist['loss'], 'b-', label='Train')
a2.plot(r, hist['val_loss'], 'r-', label='Val')
a2.set_title('Loss'); a2.set_xlabel('Época')
a2.legend(); a2.grid(alpha=0.3)
if MODELO == 'mobilenet':
    # Línea vertical separando fase 1 y fase 2
    n1 = len(h1.history['accuracy'])
    a1.axvline(x=n1 + 0.5, color='gray', linestyle='--', alpha=0.5, label='Fine-tuning')
    a2.axvline(x=n1 + 0.5, color='gray', linestyle='--', alpha=0.5)
    a1.legend()
plt.tight_layout(); plt.show()

---
## 6. Evaluar

Matriz de confusión, classification report, y galería de errores.

In [ ]:
# ============================================================
# EVALUACIÓN COMPLETA
# ============================================================

# Generador sin augmentation para evaluación limpia (mismo preprocesamiento)
eval_gen = val_datagen.flow_from_directory(
    os.path.join(cfg['split_dir'], 'validation'),
    target_size=(sz, sz), batch_size=32,
    class_mode='categorical', shuffle=False
)

loss, acc = model.evaluate(eval_gen, verbose=0)
print(f"Accuracy en validación: {acc:.1%}\n")

eval_gen.reset()
preds = model.predict(eval_gen, verbose=0)
y_pred = np.argmax(preds, axis=1)
y_true = eval_gen.classes
labels = list(eval_gen.class_indices.keys())

# Classification report
print(classification_report(y_true, y_pred,
                            target_names=labels, zero_division=0))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(5, len(labels)*1.5), max(4, len(labels)*1.2)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicción'); plt.ylabel('Real')
plt.title(f'Matriz de confusión — Accuracy: {acc:.1%}')
plt.tight_layout(); plt.show()

# Galería de errores
errs = [(i, eval_gen.filenames[i]) for i in range(len(y_true))
        if y_pred[i] != y_true[i]]
if errs:
    n = min(8, len(errs))
    print(f"\nErrores: {len(errs)} de {len(y_true)} imágenes")
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    if rows == 1 and cols == 1: axes = np.array([axes])
    for ax in np.array(axes).ravel(): ax.axis('off')
    for j, ax in enumerate(np.array(axes).ravel()):
        if j >= n: break
        idx, fname = errs[j]
        val_dir = os.path.join(cfg['split_dir'], 'validation')
        p = os.path.join(val_dir, fname)
        if os.path.exists(p):
            ax.imshow(load_img(p, target_size=(sz, sz)))
            ax.set_title(f"Real: {labels[y_true[idx]]}\n"
                         f"Pred: {labels[y_pred[idx]]} ({preds[idx][y_pred[idx]]:.0%})",
                         fontsize=9, color='red')
    plt.suptitle('Imágenes mal clasificadas', fontsize=14)
    plt.tight_layout(); plt.show()
else:
    print("\n\u2714 Sin errores en validación.")

---
## 7. Exportar modelo

Guarda el modelo en `modelos/` con un nombre descriptivo para comparar experimentos.
Puede entrenar varias veces con distintos parámetros y luego probar cada `.h5` en `probador.ipynb`.

In [ ]:
# ============================================================
# EXPORTAR MODELO
# ============================================================

import json as _json
import time as _time

os.makedirs('modelos', exist_ok=True)

# Nombre descriptivo: tipo_resolución_accuracy_fecha
va_pct = int(hist['val_accuracy'][-1] * 100)
timestamp = _time.strftime('%m%d_%H%M')
tipo_str = 'mobilenet' if MODELO == 'mobilenet' else 'cnn'
nombre = f"modelos/{tipo_str}_{sz}px_val{va_pct}_{timestamp}.h5"

model.save(nombre)
mb = os.path.getsize(nombre) / 1024 / 1024

# --- Guardar metadata JSON ---
metadata = {
    "class_names": cfg['class_names'],
    "img_size": sz,
    "preprocessing": "mobilenet" if MODELO == 'mobilenet' else "rescale",
    "model_type": tipo_str,
    "dataset_dir": cfg['dataset_dir'],
    "val_accuracy": round(float(hist['val_accuracy'][-1]), 4),
    "train_accuracy": round(float(hist['accuracy'][-1]), 4),
    "epochs": len(hist['accuracy']),
    "timestamp": _time.strftime('%Y-%m-%d %H:%M'),
}
json_path = nombre.rsplit('.', 1)[0] + '.json'
with open(json_path, 'w') as f:
    _json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✔ Modelo guardado: {nombre} ({mb:.1f} MB)")
print(f"✔ Metadata guardada: {json_path}")
print(f"  Tipo: {'MobileNetV2 (transfer learning)' if MODELO == 'mobilenet' else 'CNN custom'}")
print(f"  Val accuracy: {hist['val_accuracy'][-1]:.1%}")
print(f"  Train accuracy: {hist['accuracy'][-1]:.1%}")

# Listar todos los modelos guardados
print(f"\n--- Modelos en modelos/ ---")
for m in sorted(glob.glob('modelos/*.h5') + glob.glob('modelos/*.keras')):
    mb = os.path.getsize(m) / 1024 / 1024
    has_json = '✔' if os.path.exists(m.rsplit('.', 1)[0] + '.json') else ' '
    print(f"  [{has_json}] {m} ({mb:.1f} MB)")

print(f"\n{'='*55}")
print("PARA PROBAR: abra probador.ipynb y configure:")
print(f"{'='*55}")
print(f'MODEL_PATH = "{nombre}"')
print(f"{'='*55}")
print("  probador.ipynb detecta automáticamente clases,")
print("  preprocesamiento y dataset desde el archivo .json")

print(f"\n{'='*55}")
print("PARA scripts/prueba_video.py o inferencia_plc.py:")
print(f"{'='*55}")
print(f'MODEL_PATH = "proyecto/{nombre}"')
print(f'IMG_SIZE = {sz}')
print(f'CLASS_NAMES = {cfg["class_names"]}')
print(f'CONFIDENCE_THRESHOLD = 0.6')
print(f"{'='*55}")

if MODELO == 'mobilenet':
    print("\n⚠ Este modelo usa preprocesamiento MobileNetV2.")
    print("  prueba_video.py e inferencia_plc.py lo detectan automáticamente.")